# 02 - Generative Pretraining with Masked Autoencoder (MAE) & Latent Space Clustering

## Overview
The **Masked Autoencoder (MAE)** is a self-supervised generative paradigm adapted here for 1D multi-channel seismic waveforms. Rather than learning by contrasting pairs of traces, MAE learns representations by reconstructing missing intervals of the signal.

### How MAE Works on 1D Seismic Waveforms:
1. **1D Patchification**: The continuous $2000$-sample trace is divided into $T = 20$ non-overlapping patches (with `patch_size = 100`).
2. **High Masking Ratio ($75\%$)**: Three out of every four patches are randomly masked out. Only $25\%$ of patches (5 visible patches) + a learnable `[CLS]` token are sent to the encoder.
3. **ViT Encoder**: Standard Vision Transformer blocks operate exclusively on the unmasked visible patches, saving substantial computational resources.
4. **ViT Decoder**: A lightweight decoder receives the latent representations + learnable `[MASK]` tokens positioned at their original temporal indices, reconstructing all $2000$ timesteps.
5. **Reconstruction Loss**: Mean Squared Error (MSE) is computed strictly on the masked regions: $\mathcal{L} = \frac{1}{\text{mask\_ratio}} \| (x_{\text{pred}} - x) \odot \text{mask} \|^2$.

### Objectives in this Notebook
- Inspect the `MAE_ViT` architecture from [`Pretraining_Model/Generative_Encoder/MAE/mae/MAE.py`](../Pretraining_Model/Generative_Encoder/MAE/mae/MAE.py).
- Run or inspect pretraining with learning rate warmup and cosine decay.
- **Visualize Waveform Reconstruction**: Ground Truth vs. Masked Input vs. Model Reconstruction.
- **Latent Clustering & Manifold Projections**: Extract unlabelled representations on the test set, evaluate K-Means clustering, compute metrics (Hungarian Accuracy, ARI, NMI, Silhouette), and project embeddings to 2D using UMAP and t-SNE.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, accuracy_score, confusion_matrix
)
from scipy.optimize import linear_sum_assignment
import umap

# Setup path to import project modules
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from Config.dataset_config import extractedSTEAD
from Config.Pretraining_Models_Config import MAE
from Pretraining_Model.utility.Dataset.Dataset_STEAD import train_set, test_set, validation_set
from Pretraining_Model.Generative_Encoder.MAE.build_dataset import CustomTensorDataset
from Pretraining_Model.Generative_Encoder.MAE.mae.MAE import MAE_ViT, ViT_Classifier
from Downstream_Task.ClusteringModel_MAE import extract_embeddings, hungarian_accuracy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"MAE Configuration:\n{MAE}")

## 1. Initializing the MAE Architecture

Let's instantiate the `MAE_ViT` model using the parameters from [`Config/Pretraining_Models_Config.py`](../Config/Pretraining_Models_Config.py).
- Sample shape: `[3, 2000]` (3 channels, 2000 timesteps)
- Patch size: `100` (Number of patches = $2000 / 100 = 20$)
- Mask ratio: `0.75` (15 patches masked, 5 patches visible)
- Embedding dim: `512`
- Encoder layers: `6`, Decoder layers: `2`, Attention heads: `16`

In [ ]:
model = MAE_ViT(
    sample_shape=[extractedSTEAD['n_channel'], extractedSTEAD['n_length']],
    patch_size=MAE['patch_size'],
    mask_ratio=MAE['mask_ratio'],
    emb_dim=MAE['emb_dim'],
    encoder_layer=MAE['encoder_layer'],
    decoder_layer=MAE['decoder_layer'],
    encoder_head=MAE['attention_head'],
    decoder_head=MAE['attention_head'],
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
encoder_params = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
decoder_params = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)

print(f"MAE Total Parameters:   {total_params:,}")
print(f"Encoder Parameters:     {encoder_params:,}")
print(f"Decoder Parameters:     {decoder_params:,}")

## 2. Loading Pretrained Weights or Running Demonstration Pretraining

We can load the saved high-performing checkpoint from `Models/Good Performing Model/` or `Models/MAE/`. You can also toggle `TRAIN_DEMO_EPOCH = True` to execute a training step.

In [ ]:
TRAIN_DEMO_EPOCH = False  # Set to True if you want to run 1 training epoch live

checkpoint_paths = [
    ROOT_DIR / "Models" / "Good Performing Model" / "MAE_extractedSTEAD_0.75_100_512_6_2_16.tar",
    ROOT_DIR / "Models" / "MAE" / "MAE_extractedSTEAD_0.75_100_512_6_2_16.tar",
]

ckpt_to_load = None
for cp in checkpoint_paths:
    if cp.exists():
        ckpt_to_load = cp
        break

if ckpt_to_load:
    print(f"Loading pretrained weights from: {ckpt_to_load.name}")
    model.load_state_dict(torch.load(str(ckpt_to_load), map_location=device))
    print("Pretrained weights loaded successfully!")
else:
    print("No pre-saved checkpoint found. Model initialized with random weights.")

# Optional 1-epoch training demonstration
if TRAIN_DEMO_EPOCH:
    train_x, train_y = train_set()
    train_loader = torch.utils.data.DataLoader(
        CustomTensorDataset((train_x, train_y)),
        batch_size=128, shuffle=True
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    losses = []
    for step, (sample, _) in enumerate(train_loader):
        sample = sample.to(device)
        pred, mask = model(sample)
        loss = torch.mean((pred - sample) ** 2 * mask) / MAE['mask_ratio']
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if step % 5 == 0:
            print(f"  Step [{step}/{len(train_loader)}]  Reconstruction MSE: {loss.item():.4f}")
    print(f"Demo Epoch Mean Loss: {np.mean(losses):.4f}")

## 3. Waveform Reconstruction Visualizer

Let's test the model's ability to reconstruct missing intervals of seismic waveforms.
We pass a test trace through `model(sample)` to get:
1. **Ground Truth**: The complete original 3-channel seismogram.
2. **Masked Input**: The signal with 75% of temporal patches replaced by zeros / omitted.
3. **Reconstructed Signal**: The decoder's predicted wave arrival and coda.

In [ ]:
test_x, test_y = test_set()
model.eval()

# Select an earthquake sample from the test set
sample_idx = np.where(test_y == 1)[0][0]
sample_tensor = torch.tensor(test_x[sample_idx:sample_idx+1], dtype=torch.float32).to(device)

with torch.no_grad():
    predicted_sample, mask = model(sample_tensor)

orig_sig = sample_tensor.cpu().numpy()[0]
pred_sig = predicted_sample.cpu().numpy()[0]
mask_arr = mask.cpu().numpy()[0]
masked_input = orig_sig * (1.0 - mask_arr)

time_axis = np.linspace(0, 20.0, extractedSTEAD['n_length'])
channel_names = ['Vertical (Z)', 'North-South (N)', 'East-West (E)']

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for c in range(3):
    axes[c].plot(time_axis, orig_sig[c], label='Ground Truth Waveform', color='#2ca02c', lw=1.2, alpha=0.8)
    axes[c].plot(time_axis, masked_input[c], label='Masked Input (25% visible)', color='#1f77b4', lw=1.5)
    axes[c].plot(time_axis, pred_sig[c], label='MAE Reconstruction', color='#d62728', lw=1.2, ls='--', alpha=0.9)
    axes[c].set_ylabel(channel_names[c])
    axes[c].legend(loc='upper right')
    axes[c].grid(True, alpha=0.3)

axes[0].set_title(f"MAE 1D Seismic Waveform Reconstruction (Test Sample #{sample_idx} - Earthquake)", fontsize=13, fontweight='bold')
axes[2].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

mse_eval = np.mean((pred_sig - orig_sig)**2)
print(f"Overall Sequence Mean Squared Error (MSE): {mse_eval:.5f}")

## 4. Latent Space Clustering & Manifold Projections

How well do the unlabelled representations separate earthquakes from noise without any supervised fine-tuning?

We wrap the pretrained encoder into a `ViT_Classifier` to use `forward_features()` which extracts the `[CLS]` token representation without masking:
$$h = \text{ViT\_Classifier.forward\_features}(x) \in \mathbb{R}^{512}$$

We then:
1. Apply $L_2$-normalization: $h_{\text{norm}} = \frac{h}{\|h\|_2 + 10^{-8}}$.
2. Perform **K-Means** clustering ($k = 2$).
3. Match cluster IDs to ground-truth labels optimally using the **Hungarian Algorithm**.
4. Calculate **Clustering Accuracy**, **Adjusted Rand Index (ARI)**, **Normalized Mutual Information (NMI)**, and **Silhouette Score**.
5. Project representations to 2D using **UMAP** and **t-SNE**.

In [ ]:
test_x, test_y = test_set()
classifier = ViT_Classifier(model.encoder, num_classes=extractedSTEAD['n_class']).to(device)

# Extract embeddings on test set
embeddings = extract_embeddings(classifier, test_x, device, batch_size=128)
h_norm = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8)

# K-Means Clustering
kmeans = KMeans(n_clusters=extractedSTEAD['n_class'], n_init=20, random_state=42)
cluster_labels = kmeans.fit_predict(h_norm)

# Compute Clustering Metrics
acc, cluster_labels_remapped = hungarian_accuracy(test_y, cluster_labels, extractedSTEAD['n_class'])
ari = adjusted_rand_score(test_y, cluster_labels)
nmi = normalized_mutual_info_score(test_y, cluster_labels)
sil = silhouette_score(h_norm, cluster_labels, sample_size=min(5000, len(test_y)))

print("=== MAE Latent Space Clustering Metrics ===")
print(f"  Clustering Accuracy (Hungarian): {acc:.4f} ({acc*100:.2f}%)")
print(f"  Adjusted Rand Index (ARI):       {ari:.4f}")
print(f"  Normalized Mutual Info (NMI):    {nmi:.4f}")
print(f"  Silhouette Score:                {sil:.4f}")

## 5. 2D Manifold Visualizations (UMAP & t-SNE)

Let's project the $512$-dimensional latent embeddings down to 2 dimensions to visually evaluate the separation between earthquake signals and ambient noise.

In [ ]:
print("Computing UMAP 2D projection...")
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
umap_coords = reducer.fit_transform(h_norm)

print("Computing t-SNE 2D projection...")
tsne = TSNE(n_components=2, random_state=42, perplexity=40, max_iter=1000)
tsne_coords = tsne.fit_transform(h_norm)

# Plot UMAP and t-SNE side-by-side
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# UMAP colored by Ground Truth
scatter1 = axes[0, 0].scatter(umap_coords[:, 0], umap_coords[:, 1], c=test_y, cmap='coolwarm', alpha=0.7, s=15)
axes[0, 0].set_title("UMAP: Colored by Ground Truth Label (0: Noise, 1: EQ)", fontweight='bold')
axes[0, 0].set_xlabel("UMAP 1")
axes[0, 0].set_ylabel("UMAP 2")

# UMAP colored by K-Means Clusters
scatter2 = axes[0, 1].scatter(umap_coords[:, 0], umap_coords[:, 1], c=cluster_labels_remapped, cmap='viridis', alpha=0.7, s=15)
axes[0, 1].set_title("UMAP: Colored by K-Means Clusters", fontweight='bold')
axes[0, 1].set_xlabel("UMAP 1")
axes[0, 1].set_ylabel("UMAP 2")

# t-SNE colored by Ground Truth
scatter3 = axes[1, 0].scatter(tsne_coords[:, 0], tsne_coords[:, 1], c=test_y, cmap='coolwarm', alpha=0.7, s=15)
axes[1, 0].set_title("t-SNE: Colored by Ground Truth Label (0: Noise, 1: EQ)", fontweight='bold')
axes[1, 0].set_xlabel("t-SNE 1")
axes[1, 0].set_ylabel("t-SNE 2")

# t-SNE colored by K-Means Clusters
scatter4 = axes[1, 1].scatter(tsne_coords[:, 0], tsne_coords[:, 1], c=cluster_labels_remapped, cmap='viridis', alpha=0.7, s=15)
axes[1, 1].set_title("t-SNE: Colored by K-Means Clusters", fontweight='bold')
axes[1, 1].set_xlabel("t-SNE 1")
axes[1, 1].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

## Summary & Next Steps

In this notebook, we demonstrated:
1. The **Masked Autoencoder (MAE)** 1D architecture with $75\%$ temporal masking.
2. Waveform reconstruction recovering missing phase arrivals and coda oscillations.
3. High unsupervised clustering accuracy on the test set, proving that the masked reconstruction objective enables the ViT encoder to separate seismic events from noise without using any labels.

Next, in [`03_Pretraining_SimCLR_and_Clustering.ipynb`](03_Pretraining_SimCLR_and_Clustering.ipynb), we explore the **Contrastive Learning (SimCLR)** paradigm!